# TP Spark 2 - Prise en main des DataFrames

Ce notebook introduit l'API DataFrame de PySpark.

Objectifs :
- creer et charger des DataFrames ;
- inspecter un schema et des types ;
- selectionner, filtrer, trier et enrichir des colonnes ;
- gerer les valeurs nulles et les doublons ;
- faire des aggregations, des jointures et une requete SQL ;
- exporter un resultat.

Jeu de donnees utilise : `Data/voitures.csv`.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

spark = SparkSession.builder \
    .appName("TP_DataFrame_Intro") \
    .master("local[*]") \
    .getOrCreate()

print("Spark est pret ! Version :", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/25 13:20:15 WARN Utils: Your hostname, codespaces-d27dcd, resolves to a loopback address: 127.0.0.1; using 10.0.2.36 instead (on interface eth0)
26/03/25 13:20:15 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/25 13:20:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark est pret ! Version : 4.1.1


## 1. Charger un jeu de donnees en DataFrame

On commence par lire un fichier CSV avec en-tete et inference de schema. Cela permet a Spark d'identifier automatiquement les colonnes numeriques et textuelles.

In [2]:
df_csv = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("Data/voitures.csv")

print("Nombre de lignes lues :", df_csv.count())
df_csv.show(5, truncate=False)

Nombre de lignes lues : 10
+---+-------------+---------+-----+-----------+----------+-----+----------+
|id |marque       |categorie|prix |kilometrage|carburant |stock|remise_pct|
+---+-------------+---------+-----+-----------+----------+-----+----------+
|1  |Renault Clio |citadine |15990|42000      |essence   |4    |5         |
|2  |Peugeot 308  |berline  |18900|51000      |diesel    |3    |7         |
|3  |Tesla Model 3|berline  |39900|18000      |electrique|2    |3         |
|4  |Dacia Duster |SUV      |21400|35000      |essence   |5    |10        |
|5  |Toyota Yaris |citadine |17250|27000      |hybride   |6    |NULL      |
+---+-------------+---------+-----+-----------+----------+-----+----------+
only showing top 5 rows


## 2. Construire un DataFrame a partir de donnees locales

Cette approche est pratique pour comprendre la logique ligne/colonne sans dependre d'un fichier externe.

Ici, on definit un schema explicitement pour voir comment PySpark type les colonnes.

In [3]:
voitures_locales = [
    (101, "Citroen C3", "citadine", 14500.0, 26000),
    (102, "Hyundai Tucson", "SUV", 27900.0, 38000),
    (103, "Skoda Octavia", "berline", 23100.0, 41000),
]

schema_local = StructType([
    StructField("id", IntegerType(), False),
    StructField("modele", StringType(), False),
    StructField("categorie", StringType(), False),
    StructField("prix", DoubleType(), False),
    StructField("kilometrage", IntegerType(), False),
])

df_local = spark.createDataFrame(voitures_locales, schema=schema_local)
df_local.show()

+---+--------------+---------+-------+-----------+
| id|        modele|categorie|   prix|kilometrage|
+---+--------------+---------+-------+-----------+
|101|    Citroen C3| citadine|14500.0|      26000|
|102|Hyundai Tucson|      SUV|27900.0|      38000|
|103| Skoda Octavia|  berline|23100.0|      41000|
+---+--------------+---------+-------+-----------+



## 3. Inspecter les lignes, le schema et les types

Les operations d'inspection sont les premieres choses a faire sur un nouveau DataFrame :
- `show()` pour voir quelques lignes ;
- `printSchema()` pour comprendre la structure ;
- `dtypes` pour recuperer la liste des colonnes et leurs types.

In [4]:
df_csv.show(10, truncate=False)
df_csv.printSchema()
print("Types detectes :", df_csv.dtypes)

+---+---------------+---------+-----+-----------+----------+-----+----------+
|id |marque         |categorie|prix |kilometrage|carburant |stock|remise_pct|
+---+---------------+---------+-----+-----------+----------+-----+----------+
|1  |Renault Clio   |citadine |15990|42000      |essence   |4    |5         |
|2  |Peugeot 308    |berline  |18900|51000      |diesel    |3    |7         |
|3  |Tesla Model 3  |berline  |39900|18000      |electrique|2    |3         |
|4  |Dacia Duster   |SUV      |21400|35000      |essence   |5    |10        |
|5  |Toyota Yaris   |citadine |17250|27000      |hybride   |6    |NULL      |
|6  |BMW X3         |SUV      |45200|62000      |diesel    |1    |4         |
|7  |Renault Clio   |citadine |15990|42000      |essence   |4    |5         |
|8  |Fiat 500       |citadine |13200|NULL       |essence   |2    |8         |
|9  |Kia Niro       |SUV      |28700|31000      |hybride   |NULL |6         |
|10 |Volkswagen Golf|berline  |22100|47000      |diesel    |3   

## 4. Selectionner, renommer et convertir des colonnes

On extrait seulement les colonnes utiles pour l'analyse, on renomme `marque` en `constructeur` et on force quelques types pour montrer l'usage de `cast()`.

In [5]:
df_prepare = df_csv.select(
    F.col("id"),
    F.col("marque").alias("constructeur"),
    F.col("categorie"),
    F.col("prix").cast("double").alias("prix"),
    F.col("kilometrage").cast("int").alias("kilometrage"),
    F.col("stock").cast("int").alias("stock"),
)

df_prepare.show(truncate=False)

+---+---------------+---------+-------+-----------+-----+
|id |constructeur   |categorie|prix   |kilometrage|stock|
+---+---------------+---------+-------+-----------+-----+
|1  |Renault Clio   |citadine |15990.0|42000      |4    |
|2  |Peugeot 308    |berline  |18900.0|51000      |3    |
|3  |Tesla Model 3  |berline  |39900.0|18000      |2    |
|4  |Dacia Duster   |SUV      |21400.0|35000      |5    |
|5  |Toyota Yaris   |citadine |17250.0|27000      |6    |
|6  |BMW X3         |SUV      |45200.0|62000      |1    |
|7  |Renault Clio   |citadine |15990.0|42000      |4    |
|8  |Fiat 500       |citadine |13200.0|NULL       |2    |
|9  |Kia Niro       |SUV      |28700.0|31000      |NULL |
|10 |Volkswagen Golf|berline  |22100.0|47000      |3    |
+---+---------------+---------+-------+-----------+-----+



## 5. Filtrer, trier et limiter les resultats

On cible ici les vehicules les plus chers pour illustrer `filter()`, `orderBy()` et `limit()`.

In [6]:
df_filtre = df_prepare \
    .filter(F.col("prix") >= 20000) \
    .orderBy(F.col("prix").desc()) \
    .limit(5)

df_filtre.show(truncate=False)

+---+---------------+---------+-------+-----------+-----+
|id |constructeur   |categorie|prix   |kilometrage|stock|
+---+---------------+---------+-------+-----------+-----+
|6  |BMW X3         |SUV      |45200.0|62000      |1    |
|3  |Tesla Model 3  |berline  |39900.0|18000      |2    |
|9  |Kia Niro       |SUV      |28700.0|31000      |NULL |
|10 |Volkswagen Golf|berline  |22100.0|47000      |3    |
|4  |Dacia Duster   |SUV      |21400.0|35000      |5    |
+---+---------------+---------+-------+-----------+-----+



## 6. Creer des colonnes calculees

`withColumn()` permet d'ajouter une information derivee a partir des colonnes existantes. Ici, on calcule un prix apres remise et une tranche de kilometrage.

In [7]:
df_calcule = df_csv \
    .withColumn(
        "prix_apres_remise",
        F.round(F.col("prix") * (1 - F.coalesce(F.col("remise_pct"), F.lit(0)) / 100), 2),
    ) \
    .withColumn(
        "tranche_km",
        F.when(F.col("kilometrage") < 30000, "faible")
         .when(F.col("kilometrage") < 50000, "moyen")
         .otherwise("eleve"),
    )

df_calcule.select(
    "marque", "prix", "remise_pct", "prix_apres_remise", "kilometrage", "tranche_km"
).show(truncate=False)

+---------------+-----+----------+-----------------+-----------+----------+
|marque         |prix |remise_pct|prix_apres_remise|kilometrage|tranche_km|
+---------------+-----+----------+-----------------+-----------+----------+
|Renault Clio   |15990|5         |15190.5          |42000      |moyen     |
|Peugeot 308    |18900|7         |17577.0          |51000      |eleve     |
|Tesla Model 3  |39900|3         |38703.0          |18000      |faible    |
|Dacia Duster   |21400|10        |19260.0          |35000      |moyen     |
|Toyota Yaris   |17250|NULL      |17250.0          |27000      |faible    |
|BMW X3         |45200|4         |43392.0          |62000      |eleve     |
|Renault Clio   |15990|5         |15190.5          |42000      |moyen     |
|Fiat 500       |13200|8         |12144.0          |NULL       |eleve     |
|Kia Niro       |28700|6         |26978.0          |31000      |moyen     |
|Volkswagen Golf|22100|5         |20995.0          |47000      |moyen     |
+-----------

## 7. Gerer les valeurs nulles et les doublons

Le fichier contient volontairement quelques valeurs manquantes et une ligne dupliquee. On peut les diagnostiquer puis les traiter avec l'API `na` et `dropDuplicates()`.

In [8]:
df_nulls = df_csv.select([
    F.count(F.when(F.col(colonne).isNull(), colonne)).alias(colonne)
    for colonne in df_csv.columns
])

print("Valeurs nulles par colonne :")
df_nulls.show()

print("Nombre de lignes avant nettoyage :", df_csv.count())

df_propre = df_csv.dropDuplicates().na.fill({"stock": 0, "remise_pct": 0}).na.drop(subset=["kilometrage"])

print("Nombre de lignes apres nettoyage :", df_propre.count())
df_propre.show(truncate=False)

Valeurs nulles par colonne :
+---+------+---------+----+-----------+---------+-----+----------+
| id|marque|categorie|prix|kilometrage|carburant|stock|remise_pct|
+---+------+---------+----+-----------+---------+-----+----------+
|  0|     0|        0|   0|          1|        0|    1|         1|
+---+------+---------+----+-----------+---------+-----+----------+

Nombre de lignes avant nettoyage : 10
Nombre de lignes apres nettoyage : 9
+---+---------------+---------+-----+-----------+----------+-----+----------+
|id |marque         |categorie|prix |kilometrage|carburant |stock|remise_pct|
+---+---------------+---------+-----+-----------+----------+-----+----------+
|1  |Renault Clio   |citadine |15990|42000      |essence   |4    |5         |
|2  |Peugeot 308    |berline  |18900|51000      |diesel    |3    |7         |
|6  |BMW X3         |SUV      |45200|62000      |diesel    |1    |4         |
|10 |Volkswagen Golf|berline  |22100|47000      |diesel    |3    |5         |
|3  |Tesla Mod

## 8. Faire des aggregations et des groupements

On resume maintenant les donnees par categorie avec plusieurs fonctions d'agregation.

In [9]:
df_agreg = df_propre.groupBy("categorie").agg(
    F.count("*").alias("nb_voitures"),
    F.round(F.avg("prix"), 2).alias("prix_moyen"),
    F.sum("stock").alias("stock_total"),
    F.max("prix").alias("prix_max"),
)

df_agreg.orderBy("categorie").show()

+---------+-----------+----------+-----------+--------+
|categorie|nb_voitures|prix_moyen|stock_total|prix_max|
+---------+-----------+----------+-----------+--------+
|      SUV|          3|  31766.67|          6|   45200|
|  berline|          3|  26966.67|          8|   39900|
| citadine|          3|   16410.0|         14|   17250|
+---------+-----------+----------+-----------+--------+



## 9. Joindre plusieurs DataFrames

On ajoute ici une petite table de correspondance pour enrichir chaque categorie avec un segment commercial.

In [10]:
segments = [
    ("citadine", "urbain"),
    ("berline", "polyvalent"),
    ("SUV", "familial"),
]

df_segments = spark.createDataFrame(segments, ["categorie", "segment"])

df_jointure = df_propre.join(df_segments, on="categorie", how="left")
df_jointure.select("marque", "categorie", "segment", "prix").show(truncate=False)

+---------------+---------+----------+-----+
|marque         |categorie|segment   |prix |
+---------------+---------+----------+-----+
|Renault Clio   |citadine |urbain    |15990|
|Peugeot 308    |berline  |polyvalent|18900|
|BMW X3         |SUV      |familial  |45200|
|Volkswagen Golf|berline  |polyvalent|22100|
|Tesla Model 3  |berline  |polyvalent|39900|
|Dacia Duster   |SUV      |familial  |21400|
|Renault Clio   |citadine |urbain    |15990|
|Kia Niro       |SUV      |familial  |28700|
|Toyota Yaris   |citadine |urbain    |17250|
+---------------+---------+----------+-----+



## 10. Utiliser les requetes SQL sur un DataFrame

L'API DataFrame et SQL s'appuient sur le meme moteur. Une vue temporaire permet de passer facilement d'un style a l'autre.

In [11]:
df_propre.createOrReplaceTempView("voitures")

spark.sql("""
SELECT
    categorie,
    COUNT(*) AS nb_voitures,
    ROUND(AVG(prix), 2) AS prix_moyen,
    MAX(prix) AS prix_max
FROM voitures
GROUP BY categorie
ORDER BY prix_moyen DESC
""").show()

+---------+-----------+----------+--------+
|categorie|nb_voitures|prix_moyen|prix_max|
+---------+-----------+----------+--------+
|      SUV|          3|  31766.67|   45200|
|  berline|          3|  26966.67|   39900|
| citadine|          3|   16410.0|   17250|
+---------+-----------+----------+--------+



## 11. Ecrire un DataFrame dans un fichier

Pour garder un resultat, on peut ecrire un DataFrame vers un dossier de sortie. En Spark, une ecriture CSV cree un repertoire contenant une ou plusieurs partitions.

In [12]:
output_path = "Data/output/voitures_par_categorie"

df_agreg.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(output_path)

print("Resultat exporte dans :", output_path)

Resultat exporte dans : Data/output/voitures_par_categorie


## 12. Exercice d'application

Objectif : manipuler des DataFrames sans fichier externe.

Consignes :
1. Creer un DataFrame `df_produits` a partir de `produits`.
2. Creer un DataFrame `df_ventes` a partir de `ventes`.
3. Faire une jointure sur `id_produit`.
4. Creer une colonne `chiffre_affaires` egale a `quantite * prix_unitaire`.
5. Calculer le chiffre d'affaires total par categorie.
6. Afficher le top 3 des produits qui generent le plus de chiffre d'affaires.

In [13]:
produits = [
    (1, "Clavier", "informatique", 35.0),
    (2, "Ecran", "informatique", 180.0),
    (3, "Chaise", "mobilier", 95.0),
    (4, "Lampe", "mobilier", 28.0),
]

ventes = [
    (1, 3),
    (2, 2),
    (1, 4),
    (3, 5),
    (4, 6),
    (2, 1),
]

# 1. Creer un DataFrame `df_produits` a partir de `produits`.
df_produits = spark.createDataFrame(
    produits,
    ["id_produit", "produit", "categorie", "prix_unitaire"],
)

# 2. Creer un DataFrame `df_ventes` a partir de `ventes`.
df_ventes = spark.createDataFrame(
    ventes,
    ["id_produit", "quantite"],
)

# 3. Faire une jointure entre `df_ventes` et `df_produits`.
df_joint = df_ventes.join(df_produits, on="id_produit", how="inner")

# 4. Calculer le chiffre d'affaires pour chaque vente (quantite * prix_unitaire) et l'ajouter comme une nouvelle colonne `chiffre_affaires`.

df_ca = df_joint.withColumn(
    "chiffre_affaires",
    F.col("quantite") * F.col("prix_unitaire"),
)

# 5. Afficher le chiffre d'affaires total par categorie.

ca_par_categorie = df_ca.groupBy("categorie").agg(
    F.round(F.sum("chiffre_affaires"), 2).alias("ca_total"),
)

print("Chiffre d'affaires total par categorie :")
ca_par_categorie.show()

# 6. Afficher le top 3 des produits par chiffre d'affaires (ca_produit) en affichant le nom du produit, sa categorie et son chiffre d'affaires total.

top3 = df_ca.groupBy("produit", "categorie").agg(
    F.round(F.sum("chiffre_affaires"), 2).alias("ca_produit"),
).orderBy(F.col("ca_produit").desc()).limit(3)


print("Top 3 produits par chiffre d'affaires :")
top3.show()

Chiffre d'affaires total par categorie :


+------------+--------+
|   categorie|ca_total|
+------------+--------+
|    mobilier|   643.0|
|informatique|   785.0|
+------------+--------+

Top 3 produits par chiffre d'affaires :


+-------+------------+----------+
|produit|   categorie|ca_produit|
+-------+------------+----------+
|  Ecran|informatique|     540.0|
| Chaise|    mobilier|     475.0|
|Clavier|informatique|     245.0|
+-------+------------+----------+

